In [5]:
# Лабораторная работа: Массовый поиск подстрок с CUDA
import numpy as np
import time
import random
from numba import cuda

print("=" * 60)
print("ЛАБОРАТОРНАЯ РАБОТА: МАССОВЫЙ ПОИСК ПОДСТРОК (CUDA)")
print("=" * 60)

# Проверка GPU
if not cuda.is_available():
    print("ОШИБКА: GPU не найден")
    exit(1)

print(f"GPU: {cuda.get_current_device().name}")

# Параметры
h = 50000
num_substrings = 100
min_len = 3
max_len = 15

print(f"\nПараметры: h={h}, |N|={num_substrings}, длины={min_len}-{max_len}")

# Генерация данных
buffer = bytearray(random.randint(0, 255) for _ in range(h))
substrings = []
for i in range(num_substrings):
    length = random.randint(min_len, max_len)
    start = random.randint(0, h - length)
    substrings.append(bytes(buffer[start:start+length]))
random.shuffle(substrings)

# ============ CPU ВЕРСИЯ ============
print("\n[CPU] Выполнение...")
start = time.time()

char_map = {}
for sid, s in enumerate(substrings):
    for k, ch in enumerate(s):
        char_map.setdefault(ch, []).append((sid, k))

R = [[len(substrings[i]) for _ in range(h)] for i in range(num_substrings)]

for i, ch in enumerate(buffer):
    if ch in char_map:
        for sid, k in char_map[ch]:
            pos = i - k
            if pos >= 0:
                R[sid][pos] -= 1

cpu_results = []
for sid in range(num_substrings):
    for pos in range(h):
        if R[sid][pos] == 0:
            cpu_results.append((sid, pos))

cpu_time = time.time() - start
print(f"[CPU] Время: {cpu_time:.4f} сек")
print(f"[CPU] Найдено: {len(cpu_results)}")

# ============ GPU ВЕРСИЯ ============
print("\n[GPU] Выполнение...")

# Предварительная обработка
pairs_per_char = [[] for _ in range(256)]
for sid, s in enumerate(substrings):
    for k, ch in enumerate(s):
        pairs_per_char[ch].append((sid, k))

pair_starts = np.zeros(257, dtype=np.int32)
char_pairs = []
for ch in range(256):
    pair_starts[ch] = len(char_pairs) // 2
    for sid, k in pairs_per_char[ch]:
        char_pairs.append(sid)
        char_pairs.append(k)
pair_starts[256] = len(char_pairs) // 2
char_pairs = np.array(char_pairs, dtype=np.int32)

substr_lens = np.array([len(s) for s in substrings], dtype=np.int32)

# CUDA ядро
@cuda.jit
def kernel(buffer, char_pairs, pair_starts, result_matrix, h, n):
    idx = cuda.grid(1)
    if idx >= h:
        return
    ch = buffer[idx]
    start = pair_starts[ch]
    end = pair_starts[ch + 1]
    for p in range(start, end):
        sid = char_pairs[2 * p]
        k = char_pairs[2 * p + 1]
        pos = idx - k
        if pos >= 0:
            cuda.atomic.sub(result_matrix, sid * h + pos, 1)

# Функция для создания свежей матрицы
def create_result_matrix():
    rm = np.zeros(num_substrings * h, dtype=np.int32)
    for i in range(num_substrings):
        rm[i * h:(i + 1) * h] = substr_lens[i]
    return rm

threads = 256
blocks = (h + threads - 1) // threads

# Прогрев (с отдельной матрицей)
warmup_matrix = create_result_matrix()
d_warmup = cuda.to_device(warmup_matrix)
d_buffer = cuda.to_device(np.array(buffer, dtype=np.uint8))
d_char_pairs = cuda.to_device(char_pairs)
d_pair_starts = cuda.to_device(pair_starts)

kernel[blocks, threads](d_buffer, d_char_pairs, d_pair_starts, d_warmup, h, num_substrings)
cuda.synchronize()
del d_warmup

# Основной запуск (с новой матрицей)
result_matrix = create_result_matrix()
d_result_matrix = cuda.to_device(result_matrix)

start = time.time()
kernel[blocks, threads](d_buffer, d_char_pairs, d_pair_starts, d_result_matrix, h, num_substrings)
cuda.synchronize()
gpu_time = time.time() - start

# Сбор результатов
result_matrix = d_result_matrix.copy_to_host()
gpu_results = []
for sid in range(num_substrings):
    for pos in range(h):
        if result_matrix[sid * h + pos] == 0:
            gpu_results.append((sid, pos))

print(f"[GPU] Время: {gpu_time:.4f} сек")
print(f"[GPU] Найдено: {len(gpu_results)}")

# ============ СРАВНЕНИЕ ============
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ")
print("=" * 60)
print(f"CPU время: {cpu_time:.4f} сек")
print(f"GPU время: {gpu_time:.4f} сек")
print(f"Ускорение: {cpu_time/gpu_time:.2f}x")

cpu_set = set(cpu_results)
gpu_set = set(gpu_results)

if cpu_set == gpu_set:
    print("Совпадение результатов: ДА")
else:
    print("Совпадение результатов: НЕТ")
    print(f"Только в CPU: {len(cpu_set - gpu_set)}")
    print(f"Только в GPU: {len(gpu_set - cpu_set)}")

print(f"Всего найдено подстрок: {len(cpu_results)}")

if cpu_results:
    print(f"\nПример (подстрока, позиция): {cpu_results[:5]}")

ЛАБОРАТОРНАЯ РАБОТА: МАССОВЫЙ ПОИСК ПОДСТРОК (CUDA)
GPU: Tesla T4

Параметры: h=50000, |N|=100, длины=3-15

[CPU] Выполнение...
[CPU] Время: 1.9574 сек
[CPU] Найдено: 100

[GPU] Выполнение...
[GPU] Время: 0.0006 сек
[GPU] Найдено: 100

РЕЗУЛЬТАТЫ
CPU время: 1.9574 сек
GPU время: 0.0006 сек
Ускорение: 3546.32x
Совпадение результатов: ДА
Всего найдено подстрок: 100

Пример (подстрока, позиция): [(0, 32501), (1, 17896), (2, 27570), (3, 12836), (4, 10592)]
